# L05 · Inverse Kinematics, End-Effector Poses, and Cameras

**From a pose target to measured motion and pixels**

This lab turns one world-frame hand pose into an arm-joint candidate, validates the candidate, executes it with the joint-position PD control workflow introduced in L04, and measures the result as both a hand trajectory and camera observations. It then diagnoses an unreachable target and extends the same checks to four environments with a selective update.

## What this lab produces

- a world/base/hand/camera frame map and a w-x-y-z quaternion check;
- FK and full `(6, 9)` versus arm `(6, 7)` Jacobian evidence;
- reachable IK acceptance, 180-step dynamic execution, and separate solver/FK/tracking errors;
- fixed-camera RGB/depth and wrist-camera keyframes, or an explicit measured-state fallback;
- an unreachable best-effort candidate that is rejected before an explicit diagnostic override;
- B=4 baseline reaching and `envs_idx=[1, 3]` selective-update checks; and
- Guided interpretations generated from the current arrays.

Predict before running. A finite q, no exception, or a plausible image is not sufficient evidence.


## Before you run

The default `ROBO_GENESIS_BACKEND=auto` path selects the verified AMD backend when available and otherwise uses CPU. Use `ROBO_GENESIS_BACKEND=cpu` for the minimum path. The notebook prints both requested and actual backend.

Rendering is independent. `ROBO_GENESIS_RENDER=0` runs all IK, FK, Jacobian, control, unreachable, and batch checks without creating a camera; plots are labeled measured-state schematics, not camera frames. Set it to `1` before starting the kernel to require real fixed/wrist camera RGB, fixed-camera depth, K, fresh extrinsics, and valid clipping-range pixels. A requested render failure stops the notebook.

The setup cell calls `gs.init()` exactly once. Restart the kernel before rerunning it or changing backend/render topology. Outputs go under `ROBO_GENESIS_OUTPUTS_DIR` or the repository `outputs/` directory. No network, table, YCB asset, grasp logic, or policy is required.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.scene_config import (
    FRANKA_FORCE_MAX,
    FRANKA_FORCE_MIN,
    FRANKA_KP,
    FRANKA_KV,
    FRANKA_MJCF,
    FRANKA_QPOS,
    WRIST_CAM_FAR,
    WRIST_CAM_FOV,
    WRIST_CAM_NEAR,
    WRIST_CAM_OFFSET_EULER,
    WRIST_CAM_OFFSET_POS,
)
from genesis.utils.geom import euler_to_R, trans_R_to_T

lesson = load_course_manifest().lesson("L05")
assert lesson.slug == "inverse-kinematics-end-effector-poses-and-cameras"
assert lesson.status.value == "planned"

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")

render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l05-ik-poses-cameras", show_viewer=False)
output_dir = runtime["output_dir"]
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == "cpu" else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")

if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)

print(f"{'requested backend':>24}: {backend_mode}")
print(f"{'actual backend':>24}: {actual_backend}")
print(f"{'render enabled':>24}: {render_enabled}")
print(f"{'output directory':>24}: {output_dir.resolve()}")


## Predict before building

Write down your answers first:

1. Does a finite `(9,)` q prove that IK converged?
2. Does a residual below tolerance prove that the dynamic hand reached its target?
3. What orientation error should `q_target` and `-q_target` produce?
4. Which transform should remain fixed, and which should change after the hand moves?
5. For `res=(640, 360)`, what are the RGB and depth shapes?
6. Can one camera frame prove all four environments passed?

The answers must come from frame conventions, residuals, measured trajectories, and per-environment arrays—not from the absence of an exception.


In [ ]:
DT = 0.01
SUBSTEPS = 2
REACH_STEPS = 180
IK_POSITION_TOLERANCE = 5e-4
IK_ROTATION_TOLERANCE = 5e-3
EXECUTE_REJECTED_IK_FOR_DIAGNOSTIC = True
LIMIT_TOLERANCE = 1e-6

TARGET_POSITION = np.array([0.45, 0.0, 0.35], dtype=float)
TARGET_QUATERNION = np.array([0.0, 1.0, 0.0, 0.0], dtype=float)
UNREACHABLE_POSITION = np.array([2.0, 0.0, 2.0], dtype=float)
FIXED_CAMERA_RESOLUTION = (640, 360)
FIXED_CAMERA_FOV = 45.0
FIXED_CAMERA_NEAR = 0.05
FIXED_CAMERA_FAR = 10.0
WRIST_CAMERA_RESOLUTION = (640, 360)

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=DT, substeps=SUBSTEPS),
    rigid_options=gs.options.RigidOptions(enable_collision=True),
    show_viewer=False,
)
scene.add_entity(gs.morphs.Plane())
franka = scene.add_entity(gs.morphs.MJCF(file=FRANKA_MJCF))
target_marker = scene.add_entity(
    gs.morphs.Sphere(
        radius=0.025,
        pos=tuple(TARGET_POSITION),
        fixed=True,
        collision=False,
    ),
    surface=gs.surfaces.Default(color=(0.95, 0.20, 0.20, 1.0)),
)
reference_marker = scene.add_entity(
    gs.morphs.Box(
        size=(0.12, 0.12, 0.03),
        pos=(0.45, -0.12, 0.015),
        fixed=True,
        collision=False,
    ),
    surface=gs.surfaces.Default(color=(0.15, 0.70, 0.85, 1.0)),
)

fixed_camera = None
wrist_camera = None
if render_enabled:
    fixed_camera = scene.add_camera(
        res=FIXED_CAMERA_RESOLUTION,
        pos=(1.2, -1.2, 1.0),
        lookat=(0.35, 0.0, 0.35),
        fov=FIXED_CAMERA_FOV,
        near=FIXED_CAMERA_NEAR,
        far=FIXED_CAMERA_FAR,
        GUI=False,
    )
    wrist_camera = scene.add_camera(
        res=WRIST_CAMERA_RESOLUTION,
        fov=WRIST_CAM_FOV,
        near=WRIST_CAM_NEAR,
        far=WRIST_CAM_FAR,
        GUI=False,
    )

scene.build()
hand = franka.get_link("hand")

JOINT_NAMES = [f"joint{i}" for i in range(1, 8)] + [
    "finger_joint1",
    "finger_joint2",
]
dof_indices = []
for name in JOINT_NAMES:
    joint = franka.get_joint(name)
    if joint.n_dofs != 1 or joint.n_qs != 1:
        raise AssertionError(
            f"{name}: expected one DOF and one qpos, got "
            f"n_dofs={joint.n_dofs}, n_qs={joint.n_qs}"
        )
    dof_indices.extend(joint.dofs_idx_local)

all_dofs = np.asarray(dof_indices, dtype=int)
arm_dofs = all_dofs[:7]
finger_dofs = all_dofs[7:]
q_start = np.asarray(FRANKA_QPOS, dtype=float)
kp = np.asarray(FRANKA_KP, dtype=float)
kv = np.asarray(FRANKA_KV, dtype=float)
force_lower = np.asarray(FRANKA_FORCE_MIN, dtype=float)
force_upper = np.asarray(FRANKA_FORCE_MAX, dtype=float)

franka.set_dofs_kp(kp, dofs_idx_local=all_dofs)
franka.set_dofs_kv(kv, dofs_idx_local=all_dofs)
franka.set_dofs_force_range(force_lower, force_upper, dofs_idx_local=all_dofs)
franka.set_dofs_position(q_start, dofs_idx_local=all_dofs, zero_velocity=True)

hand_to_camera = trans_R_to_T(
    np.asarray(WRIST_CAM_OFFSET_POS, dtype=np.float64),
    euler_to_R(np.asarray(WRIST_CAM_OFFSET_EULER, dtype=np.float64)),
)
if render_enabled:
    wrist_camera.attach(hand, offset_T=hand_to_camera)
    wrist_camera.move_to_attach()
    print("render path: fixed and wrist cameras declared before build; wrist attached after build")
else:
    print("render path: SKIP — ROBO_GENESIS_RENDER=0; no camera was created")

print(
    f"built Franka: links={len(franka.links)}, joints={len(franka.joints)}, "
    f"dofs={franka.n_dofs}, qpos={franka.n_qs}"
)
print(f"outer dt={DT:.3f} s; substeps={SUBSTEPS}; internal dt={DT / SUBSTEPS:.4f} s")


## Part A · Build the frame-to-motion chain

### Scene, model, and frame map

The next cell directly declares a Plane, Franka, reachable target marker, nearby visual reference, and—only when requested—both cameras before `build()`. It configures the same nine named DOFs used in L04, then attaches the wrist camera to the built `hand` Link.

In this scene W and B coincide only because the default fixed-base Franka pose is used. `TARGET_POSITION`, `TARGET_QUATERNION`, and `hand.get_pos/get_quat(relative=False)` are world-frame quantities. `hand_to_camera` is the fixed `T_HC` mounting transform. Genesis camera extrinsics map world coordinates into camera coordinates.


In [ ]:
def read_dofs(entity, indices):
    values = to_numpy(entity.get_dofs_position(dofs_idx_local=indices)).astype(float)
    if values.shape[-1] != len(indices) or not np.isfinite(values).all():
        raise AssertionError(f"invalid DOF state: shape={values.shape}")
    return values.copy()


def read_hand_pose(link, envs_idx=None):
    position = to_numpy(link.get_pos(envs_idx=envs_idx, relative=False)).astype(float)
    quaternion = to_numpy(link.get_quat(envs_idx=envs_idx, relative=False)).astype(float)
    if position.shape[-1] != 3 or quaternion.shape[-1] != 4:
        raise AssertionError(
            f"invalid hand pose shapes: position={position.shape}, quaternion={quaternion.shape}"
        )
    if not np.isfinite(position).all() or not np.isfinite(quaternion).all():
        raise AssertionError("hand pose contains non-finite values")
    return position.copy(), quaternion.copy()


initial_q = read_dofs(franka, all_dofs).reshape(-1)
initial_qdot = to_numpy(
    franka.get_dofs_velocity(dofs_idx_local=all_dofs)
).reshape(-1).astype(float)
limit_lower, limit_upper = (
    to_numpy(value).reshape(-1).astype(float)
    for value in franka.get_dofs_limit(dofs_idx_local=all_dofs)
)
initial_hand_position, initial_hand_quaternion = (
    value.reshape(-1) for value in read_hand_pose(hand)
)

structure_checks = {
    "whole_q_shape": initial_q.shape == (9,),
    "velocity_shape": initial_qdot.shape == (9,),
    "seven_arm_two_finger": arm_dofs.shape == (7,) and finger_dofs.shape == (2,),
    "unique_local_dofs": np.array_equal(np.sort(all_dofs), np.arange(9)),
    "initial_q_matches": np.allclose(initial_q, q_start, rtol=0.0, atol=1e-6),
    "initial_velocity_zero": np.allclose(initial_qdot, 0.0, rtol=0.0, atol=1e-7),
    "limits_shape": limit_lower.shape == (9,) and limit_upper.shape == (9,),
    "initial_q_inside_limits": (
        np.all(initial_q >= limit_lower - LIMIT_TOLERANCE)
        and np.all(initial_q <= limit_upper + LIMIT_TOLERANCE)
    ),
}
if not all(structure_checks.values()):
    raise AssertionError(structure_checks)

frame_rows = [
    ("TARGET_POSITION / TARGET_QUATERNION", "world W", "desired hand-link-origin pose"),
    ("hand.get_pos/get_quat(relative=False)", "world W", "measured hand-link-origin pose"),
    ("Franka base", "base B", "coincides with W only in this scene"),
    ("fixed camera pos/lookat", "world W", "constant configured viewpoint"),
    ("hand_to_camera", "hand H → camera C", "constant wrist mounting offset"),
    ("camera extrinsics", "world W → camera C", "world-to-camera coordinate map"),
]
lines = [
    "| Quantity | Frame | Meaning |",
    "|---|---|---|",
    *[f"| {quantity} | {frame} | {meaning} |" for quantity, frame, meaning in frame_rows],
]
display(Markdown("\n".join(lines)))
print("structure checks:", structure_checks)
print("initial hand position [m]:", initial_hand_position)
print("initial hand quaternion [wxyz]:", initial_hand_quaternion)


### Read the quaternion, FK, and Jacobian logic

A pose contains position and orientation. Genesis uses w-x-y-z unit quaternions, and `q`/`-q` represent the same rotation. The next cell exposes normalization and shortest-angle error rather than hiding them in a helper module.

FK predicts link poses for a supplied q but does not move the scene. In Genesis 1.3.3, `forward_kinematics()` reuses scratch storage that is lazily allocated by the first IK call, so this notebook evaluates FK immediately after that initialization. This version-specific call order does not turn FK into dynamic control. The solver-state check is read immediately after IK, before FK. It uses a normalized, sign-aware quaternion component distance because `arccos` amplifies float32 rounding near zero; target and trajectory orientation errors still use the shortest angular distance. The spatial Jacobian maps qdot to `[v; omega]` locally. The full Franka matrix is `(6, 9)`; selecting the seven named arm columns gives `(6, 7)`. Its singular values describe only this sampled configuration and are not a universal singularity test.

The same cell also exposes camera K, live world-to-camera extrinsics, and strict RGB/depth validation. In Genesis 1.3.3, `extrinsics` is cached, so the moving wrist comparison deliberately recomputes it from the live public `camera.transform` using the pinned axis convention.


In [ ]:
def normalize_wxyz(quaternion):
    quaternion = np.asarray(quaternion, dtype=float)
    if quaternion.shape != (4,) or not np.isfinite(quaternion).all():
        raise ValueError("quaternion must be a finite wxyz vector of shape (4,)")
    norm = float(np.linalg.norm(quaternion))
    if norm < 1e-12:
        raise ValueError("a zero quaternion does not define an orientation")
    return quaternion / norm


def quaternion_angle_error(measured_wxyz, target_wxyz):
    measured = normalize_wxyz(measured_wxyz)
    target = normalize_wxyz(target_wxyz)
    cosine_half_angle = np.clip(abs(np.dot(measured, target)), 0.0, 1.0)
    return float(2.0 * np.arccos(cosine_half_angle))


def quaternion_component_distance(first_wxyz, second_wxyz):
    first = normalize_wxyz(first_wxyz)
    second = normalize_wxyz(second_wxyz)
    return float(
        min(
            np.max(np.abs(first - second)),
            np.max(np.abs(first + second)),
        )
    )


def pinhole_intrinsics(resolution, vertical_fov_degrees):
    width, height = resolution
    focal = 0.5 * height / np.tan(np.deg2rad(0.5 * vertical_fov_degrees))
    return np.array(
        [[focal, 0.0, 0.5 * width], [0.0, focal, 0.5 * height], [0.0, 0.0, 1.0]],
        dtype=float,
    )


def live_world_to_camera(camera):
    world_from_graphics_camera = np.asarray(camera.transform, dtype=float)
    graphics_to_camera_axes = np.diag([1.0, -1.0, -1.0, 1.0])
    world_from_camera = world_from_graphics_camera @ graphics_to_camera_axes
    return np.linalg.inv(world_from_camera)


def render_observation(camera, label, *, include_depth):
    rgb, depth, _, _ = camera.render(
        rgb=True,
        depth=include_depth,
        segmentation=False,
        normal=False,
    )
    rgb = to_numpy(rgb)
    width, height = camera.res
    if rgb.shape != (height, width, 3) or rgb.dtype != np.uint8:
        raise AssertionError(
            f"{label}: RGB must be uint8 {(height, width, 3)}, got {rgb.dtype} {rgb.shape}"
        )
    if rgb.size == 0 or not np.isfinite(rgb).all():
        raise AssertionError(f"{label}: RGB is empty or non-finite")
    print(
        f"{label}: RGB shape={rgb.shape}, dtype={rgb.dtype}, "
        f"range=[{int(rgb.min())}, {int(rgb.max())}]"
    )
    if not include_depth:
        return rgb.copy(), None

    depth = to_numpy(depth)
    if depth.shape != (height, width) or not np.issubdtype(depth.dtype, np.floating):
        raise AssertionError(
            f"{label}: depth must be floating {(height, width)}, got {depth.dtype} {depth.shape}"
        )
    if not np.isfinite(depth).all():
        raise AssertionError(f"{label}: depth contains non-finite values")
    valid = (depth > camera.near) & (depth < camera.far * (1.0 - 1e-3))
    if not np.any(valid):
        raise AssertionError(f"{label}: no depth pixel lies inside the clipping interval")
    valid_depth = depth[valid]
    print(
        f"{label}: depth shape={depth.shape}, dtype={depth.dtype}, "
        f"valid={int(valid.sum())}/{depth.size}, "
        f"valid range=[{float(valid_depth.min()):.6f}, {float(valid_depth.max()):.6f}] m"
    )
    return rgb.copy(), depth.copy()


target_quaternion = normalize_wxyz(TARGET_QUATERNION)
opposite_quaternion_error = quaternion_angle_error(
    target_quaternion,
    -target_quaternion,
)
assert np.isclose(opposite_quaternion_error, 0.0, atol=1e-12)

full_jacobian = to_numpy(franka.get_jacobian(hand)).astype(float)
arm_jacobian = full_jacobian[:, arm_dofs]
arm_singular_values = np.linalg.svd(arm_jacobian, compute_uv=False)
kinematic_checks = {
    "target_quaternion_unit": np.isclose(np.linalg.norm(target_quaternion), 1.0),
    "quaternion_double_cover": np.isclose(opposite_quaternion_error, 0.0, atol=1e-12),
    "full_jacobian_shape": full_jacobian.shape == (6, 9),
    "arm_jacobian_shape": arm_jacobian.shape == (6, 7),
    "jacobian_finite": np.isfinite(arm_singular_values).all(),
}
if not all(kinematic_checks.values()):
    raise AssertionError(kinematic_checks)

fixed_initial_rgb = None
wrist_initial_rgb = None
fixed_intrinsics = pinhole_intrinsics(
    FIXED_CAMERA_RESOLUTION,
    FIXED_CAMERA_FOV,
)
wrist_intrinsics = pinhole_intrinsics(
    WRIST_CAMERA_RESOLUTION,
    WRIST_CAM_FOV,
)
fixed_extrinsics_initial = None
wrist_extrinsics_initial = None
if render_enabled:
    fixed_intrinsics = np.asarray(fixed_camera.intrinsics, dtype=float)
    wrist_intrinsics = np.asarray(wrist_camera.intrinsics, dtype=float)
    fixed_extrinsics_initial = live_world_to_camera(fixed_camera)
    wrist_extrinsics_initial = live_world_to_camera(wrist_camera)
    assert np.allclose(fixed_camera.extrinsics, fixed_extrinsics_initial)
    assert np.allclose(wrist_camera.extrinsics, wrist_extrinsics_initial)
    fixed_initial_rgb, _ = render_observation(
        fixed_camera,
        "fixed camera initial",
        include_depth=False,
    )
    wrist_initial_rgb, _ = render_observation(
        wrist_camera,
        "wrist camera initial",
        include_depth=False,
    )
else:
    print("SKIP — camera creation, attachment, RGB/depth, and extrinsics checks are disabled")
    print("planned fixed K:\n", fixed_intrinsics)
    print("planned wrist K:\n", wrist_intrinsics)

print("target quaternion [wxyz]:", target_quaternion)
print("q versus -q angular error [rad]:", opposite_quaternion_error)
print("full Jacobian shape:", full_jacobian.shape)
print("arm Jacobian shape:", arm_jacobian.shape)
print("arm Jacobian singular values at the initial configuration:", arm_singular_values)
print("kinematic checks:", kinematic_checks)


### Accept an IK candidate before controlling it

Genesis can return its best finite candidate even when residual tolerances are not met. The next cell checks q `(9,)`, residual `(6,)`, finiteness, runtime joint limits, unchanged fingers, and the enabled position/rotation residual norms separately.

It also uses FK to predict the candidate hand pose and confirms that solving IK did not change scene state. A candidate is not called executed until the following cell sends it through `control_dofs_position(...)` and advances dynamics.


In [ ]:
POSITION_MASK = np.array([True, True, True])
ROTATION_MASK = np.array([True, True, True])


def assess_single_ik(candidate, residual, reference_fingers):
    candidate = to_numpy(candidate).astype(float)
    residual = to_numpy(residual).astype(float)
    shape_ok = candidate.shape == (9,) and residual.shape == (6,)
    checks = {
        "q_shape": candidate.shape == (9,),
        "residual_shape": residual.shape == (6,),
        "q_finite": np.isfinite(candidate).all(),
        "residual_finite": np.isfinite(residual).all(),
    }
    if shape_ok:
        position_norm = float(np.linalg.norm(residual[:3][POSITION_MASK]))
        rotation_norm = float(np.linalg.norm(residual[3:][ROTATION_MASK]))
        checks.update(
            {
                "joint_limits": (
                    np.all(candidate >= limit_lower - LIMIT_TOLERANCE)
                    and np.all(candidate <= limit_upper + LIMIT_TOLERANCE)
                ),
                "fingers_preserved": np.allclose(
                    candidate[finger_dofs],
                    reference_fingers,
                    rtol=0.0,
                    atol=1e-7,
                ),
                "position_residual": position_norm <= IK_POSITION_TOLERANCE,
                "rotation_residual": rotation_norm <= IK_ROTATION_TOLERANCE,
            }
        )
    else:
        position_norm = np.nan
        rotation_norm = np.nan
    return {
        "candidate": candidate,
        "residual": residual,
        "position_norm": position_norm,
        "rotation_norm": rotation_norm,
        "checks": checks,
        "valid": bool(shape_ok and all(checks.values())),
    }


q_before_ik_raw = franka.get_qpos()
q_before_ik = read_dofs(franka, all_dofs).reshape(-1)
hand_before_ik_position, hand_before_ik_quaternion = (
    value.reshape(-1) for value in read_hand_pose(hand)
)
q_goal_raw, ik_error_raw = franka.inverse_kinematics(
    link=hand,
    pos=TARGET_POSITION,
    quat=target_quaternion,
    init_qpos=q_before_ik,
    respect_joint_limit=True,
    max_samples=50,
    max_solver_iters=20,
    damping=0.01,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    pos_mask=POSITION_MASK.tolist(),
    rot_mask=ROTATION_MASK.tolist(),
    dofs_idx_local=arm_dofs,
    return_error=True,
)
reachable_ik = assess_single_ik(
    q_goal_raw,
    ik_error_raw,
    q_before_ik[finger_dofs],
)
q_goal = reachable_ik["candidate"]
q_after_ik = read_dofs(franka, all_dofs).reshape(-1)
hand_after_ik_position, hand_after_ik_quaternion = (
    value.reshape(-1) for value in read_hand_pose(hand)
)
solver_orientation_component_distance = quaternion_component_distance(
    hand_after_ik_quaternion,
    hand_before_ik_quaternion,
)
solver_state_checks = {
    "q_restored_after_ik": np.allclose(q_after_ik, q_before_ik, rtol=0.0, atol=1e-7),
    "hand_position_restored_after_ik": np.allclose(
        hand_after_ik_position,
        hand_before_ik_position,
        rtol=0.0,
        atol=1e-7,
    ),
    "hand_orientation_restored_after_ik": (
        solver_orientation_component_distance < 1e-6
    ),
}

initial_fk_positions_raw, initial_fk_quaternions_raw = franka.forward_kinematics(
    q_before_ik_raw,
    links_idx_local=[hand.idx_local],
)
initial_fk_position = to_numpy(initial_fk_positions_raw).reshape(-1).astype(float)
initial_fk_quaternion = to_numpy(initial_fk_quaternions_raw).reshape(-1).astype(float)
q_after_initial_fk = read_dofs(franka, all_dofs).reshape(-1)
initial_fk_position_error = float(
    np.linalg.norm(initial_fk_position - hand_before_ik_position)
)
initial_fk_orientation_error = quaternion_angle_error(
    initial_fk_quaternion,
    hand_before_ik_quaternion,
)
initial_fk_orientation_component_distance = quaternion_component_distance(
    initial_fk_quaternion,
    hand_before_ik_quaternion,
)
kinematic_checks.update(
    {
        "fk_position_matches_state": initial_fk_position_error < 1e-6,
        "fk_orientation_matches_state": (
            initial_fk_orientation_component_distance < 1e-6
        ),
        "fk_did_not_change_q": np.allclose(
            q_before_ik,
            q_after_initial_fk,
            rtol=0.0,
            atol=1e-7,
        ),
    }
)
fk_goal_positions_raw, fk_goal_quaternions_raw = franka.forward_kinematics(
    q_goal_raw,
    links_idx_local=[hand.idx_local],
)
fk_goal_position = to_numpy(fk_goal_positions_raw).reshape(-1).astype(float)
fk_goal_quaternion = to_numpy(fk_goal_quaternions_raw).reshape(-1).astype(float)
fk_goal_position_error = float(np.linalg.norm(fk_goal_position - TARGET_POSITION))
fk_goal_orientation_error = quaternion_angle_error(
    fk_goal_quaternion,
    target_quaternion,
)

print("reachable q shape:", q_goal.shape)
print("reachable residual shape:", reachable_ik["residual"].shape)
print("reachable position residual [m]:", reachable_ik["position_norm"])
print("reachable rotation residual [rad]:", reachable_ik["rotation_norm"])
print("candidate checks:", reachable_ik["checks"])
print("solver preserved scene state:", solver_state_checks)
print("solver orientation component distance:", solver_orientation_component_distance)
print("initial FK orientation component distance:", initial_fk_orientation_component_distance)
print("FK prediction position error [m]:", fk_goal_position_error)
print("FK prediction orientation error [rad]:", fk_goal_orientation_error)
if not reachable_ik["valid"]:
    raise AssertionError("reachable IK candidate rejected: " + str(reachable_ik["checks"]))
if not all(solver_state_checks.values()):
    raise AssertionError("IK changed dynamic scene state: " + str(solver_state_checks))


In [ ]:
reachable_initial_position, reachable_initial_quaternion = (
    value.reshape(-1) for value in read_hand_pose(hand)
)
reachable_initial_position_error = float(
    np.linalg.norm(reachable_initial_position - TARGET_POSITION)
)
reachable_initial_orientation_error = quaternion_angle_error(
    reachable_initial_quaternion,
    target_quaternion,
)

reachable_q_history = [read_dofs(franka, all_dofs).reshape(-1)]
reachable_position_history = [reachable_initial_position]
reachable_quaternion_history = [reachable_initial_quaternion]
for _ in range(REACH_STEPS):
    franka.control_dofs_position(q_goal, dofs_idx_local=all_dofs)
    scene.step()
    if render_enabled:
        wrist_camera.move_to_attach()
    measured_position, measured_quaternion = read_hand_pose(hand)
    reachable_q_history.append(read_dofs(franka, all_dofs).reshape(-1))
    reachable_position_history.append(measured_position.reshape(-1))
    reachable_quaternion_history.append(measured_quaternion.reshape(-1))

reachable_q_history = np.asarray(reachable_q_history, dtype=float)
reachable_position_history = np.asarray(reachable_position_history, dtype=float)
reachable_quaternion_history = np.asarray(reachable_quaternion_history, dtype=float)
reachable_time = np.arange(REACH_STEPS + 1, dtype=float) * DT
reachable_position_errors = np.linalg.norm(
    reachable_position_history - TARGET_POSITION,
    axis=1,
)
reachable_orientation_errors = np.array(
    [
        quaternion_angle_error(quaternion, target_quaternion)
        for quaternion in reachable_quaternion_history
    ],
    dtype=float,
)
reachable_final_position_error = float(reachable_position_errors[-1])
reachable_final_orientation_error = float(reachable_orientation_errors[-1])

execution_checks = {
    "q_history_shape": reachable_q_history.shape == (REACH_STEPS + 1, 9),
    "position_history_shape": reachable_position_history.shape == (REACH_STEPS + 1, 3),
    "quaternion_history_shape": reachable_quaternion_history.shape == (REACH_STEPS + 1, 4),
    "trajectory_finite": (
        np.isfinite(reachable_q_history).all()
        and np.isfinite(reachable_position_history).all()
        and np.isfinite(reachable_quaternion_history).all()
    ),
    "position_error_decreased": (
        reachable_final_position_error < reachable_initial_position_error
    ),
    "orientation_improved_or_initially_satisfied": (
        reachable_initial_orientation_error <= IK_ROTATION_TOLERANCE
        or reachable_final_orientation_error < reachable_initial_orientation_error
    ),
    "position_threshold": reachable_final_position_error < 0.02,
    "orientation_threshold": reachable_final_orientation_error < 0.05,
}
if not all(execution_checks.values()):
    raise AssertionError(execution_checks)

fixed_reached_rgb = None
fixed_reached_depth = None
wrist_reached_rgb = None
fixed_extrinsics_final = None
wrist_extrinsics_final = None
camera_checks = {"render_branch_explicit": True}
if render_enabled:
    fixed_extrinsics_final = live_world_to_camera(fixed_camera)
    wrist_extrinsics_final = live_world_to_camera(wrist_camera)
    fixed_reached_rgb, fixed_reached_depth = render_observation(
        fixed_camera,
        "fixed camera reached",
        include_depth=True,
    )
    wrist_reached_rgb, _ = render_observation(
        wrist_camera,
        "wrist camera reached",
        include_depth=False,
    )
    camera_checks.update(
        {
            "fixed_K_shape_and_finite": (
                fixed_intrinsics.shape == (3, 3) and np.isfinite(fixed_intrinsics).all()
            ),
            "wrist_K_shape_and_finite": (
                wrist_intrinsics.shape == (3, 3) and np.isfinite(wrist_intrinsics).all()
            ),
            "fixed_extrinsics_shape_and_finite": (
                fixed_extrinsics_final.shape == (4, 4)
                and np.isfinite(fixed_extrinsics_final).all()
            ),
            "wrist_extrinsics_shape_and_finite": (
                wrist_extrinsics_final.shape == (4, 4)
                and np.isfinite(wrist_extrinsics_final).all()
            ),
            "fixed_extrinsics_unchanged": np.allclose(
                fixed_extrinsics_final,
                fixed_extrinsics_initial,
                rtol=0.0,
                atol=1e-8,
            ),
            "wrist_extrinsics_changed": not np.allclose(
                wrist_extrinsics_final,
                wrist_extrinsics_initial,
                rtol=0.0,
                atol=1e-5,
            ),
        }
    )
else:
    camera_checks["render_disabled_has_no_camera_arrays"] = all(
        value is None
        for value in (
            fixed_initial_rgb,
            wrist_initial_rgb,
            fixed_reached_rgb,
            fixed_reached_depth,
            wrist_reached_rgb,
        )
    )

if not all(camera_checks.values()):
    raise AssertionError(camera_checks)

print("initial execution position error [m]:", reachable_initial_position_error)
print("final execution position error [m]:", reachable_final_position_error)
print("initial execution orientation error [rad]:", reachable_initial_orientation_error)
print("final execution orientation error [rad]:", reachable_final_orientation_error)
print("execution checks:", execution_checks)
print("camera checks:", camera_checks)


In [ ]:
if render_enabled:
    camera_figure, camera_axes = plt.subplots(2, 3, figsize=(15, 8))
    camera_axes[0, 0].imshow(fixed_initial_rgb)
    camera_axes[0, 0].set_title("Fixed camera — initial RGB")
    camera_axes[0, 1].imshow(fixed_reached_rgb)
    camera_axes[0, 1].set_title("Fixed camera — reached RGB")
    valid_depth = (
        (fixed_reached_depth > fixed_camera.near)
        & (fixed_reached_depth < fixed_camera.far * (1.0 - 1e-3))
    )
    depth_for_display = np.ma.masked_where(~valid_depth, fixed_reached_depth)
    camera_axes[0, 2].imshow(depth_for_display, cmap="viridis")
    camera_axes[0, 2].set_title("Fixed camera — valid depth [m]")
    camera_axes[1, 0].imshow(wrist_initial_rgb)
    camera_axes[1, 0].set_title("Wrist camera — initial RGB")
    camera_axes[1, 1].imshow(wrist_reached_rgb)
    camera_axes[1, 1].set_title("Wrist camera — reached RGB")
    camera_axes[1, 2].axis("off")
    camera_axes[1, 2].text(
        0.02,
        0.75,
        "Fixed extrinsics unchanged:\n"
        f"{camera_checks['fixed_extrinsics_unchanged']}\n\n"
        "Wrist extrinsics changed:\n"
        f"{camera_checks['wrist_extrinsics_changed']}",
        va="top",
        fontsize=11,
    )
    for axis in camera_axes.flat[:5]:
        axis.axis("off")
    camera_figure.suptitle("Genesis camera observations — fixed and hand-attached views")
    camera_figure.tight_layout()
    camera_path = output_dir / "l05_camera_keyframes.png"
    camera_figure.savefig(camera_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(camera_figure)
    print("saved:", camera_path.resolve())
else:
    fallback_figure = plt.figure(figsize=(8, 6))
    fallback_axis = fallback_figure.add_subplot(111, projection="3d")
    fallback_axis.plot(
        reachable_position_history[:, 0],
        reachable_position_history[:, 1],
        reachable_position_history[:, 2],
        color="#247BA0",
        label="measured hand trajectory",
    )
    fallback_axis.scatter(
        *reachable_position_history[0],
        color="#E9C46A",
        s=60,
        label="initial hand",
    )
    fallback_axis.scatter(
        *reachable_position_history[-1],
        color="#2A9D5B",
        s=60,
        label="final hand",
    )
    fallback_axis.scatter(*TARGET_POSITION, color="#D95F43", s=90, marker="*", label="target")
    fallback_axis.set(
        xlabel="world x [m]",
        ylabel="world y [m]",
        zlabel="world z [m]",
        title="Measured-state schematic — not a camera frame",
    )
    fallback_axis.legend()
    fallback_figure.tight_layout()
    fallback_path = output_dir / "l05_reachable_measured_state.png"
    fallback_figure.savefig(fallback_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fallback_figure)
    print("SKIP — rendering disabled before build")
    print("saved measured-state schematic:", fallback_path.resolve())

error_figure, error_axes = plt.subplots(1, 2, figsize=(12, 4))
error_axes[0].plot(reachable_time, reachable_position_errors, color="#247BA0")
error_axes[0].axhline(0.02, color="#D95F43", linestyle="--", label="0.02 m threshold")
error_axes[0].set(
    xlabel="simulated time [s]",
    ylabel="position error [m]",
    title="Measured hand-position error",
)
error_axes[1].plot(reachable_time, reachable_orientation_errors, color="#2A9D5B")
error_axes[1].axhline(0.05, color="#D95F43", linestyle="--", label="0.05 rad threshold")
error_axes[1].set(
    xlabel="simulated time [s]",
    ylabel="orientation error [rad]",
    title="Measured shortest-angle orientation error",
)
for axis in error_axes:
    axis.grid(alpha=0.25)
    axis.legend()
error_figure.tight_layout()
error_path = output_dir / "l05_reachable_pose_errors.png"
error_figure.savefig(error_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(error_figure)
print("saved:", error_path.resolve())


In [ ]:
jacobian_text = ", ".join(f"{value:.4f}" for value in arm_singular_values)
camera_text = (
    "Genesis rendering ran. Fixed RGB/depth and wrist RGB passed shape, dtype, "
    "finiteness, and clipping checks; fixed extrinsics stayed constant and fresh "
    "wrist extrinsics changed after move_to_attach()."
    if render_enabled
    else
    "Rendering was disabled before build. No RGB, depth, or runtime camera "
    "extrinsics were tested; the displayed 3-D view is a measured-state schematic."
)
reachable_interpretation = f"""
### Guided interpretation

**1. Frames and representation.** Target and measured hand poses are both in
world frame W. The target quaternion norm is
{np.linalg.norm(target_quaternion):.6f} in Genesis w-x-y-z order, and its sign-
flipped representation has {opposite_quaternion_error:.3e} rad angular error.

**2. Solver and FK.** The enabled IK residual norms are
{reachable_ik['position_norm']:.6f} m and
{reachable_ik['rotation_norm']:.6f} rad. Shape, finiteness, limits, fingers, and
both tolerances produce `ik_valid={reachable_ik['valid']}`. FK predicts errors of
{fk_goal_position_error:.6f} m and {fk_goal_orientation_error:.6f} rad; this is
kinematic evidence, not a control result.

**3. Dynamic execution and the local Jacobian.** Measured position error changed
from {reachable_initial_position_error:.6f} to
{reachable_final_position_error:.6f} m, while shortest-angle orientation error
changed from {reachable_initial_orientation_error:.6f} to
{reachable_final_orientation_error:.6f} rad over {REACH_STEPS} outer steps. The
arm Jacobian is {arm_jacobian.shape}, with singular values [{jacobian_text}].
Those values describe only the sampled initial configuration.

**4. Camera evidence.** {camera_text}

All numbers above came from this run. Solver residual, FK prediction, dynamic
tracking, and pixels remain separate evidence.
"""
display(Markdown(reachable_interpretation))


## Part B · Reject an unreachable target, then diagnose it explicitly

The target `[2, 0, 2] m` is processed by the same arm-only IK and acceptance logic. Normal control rejects it when either enabled residual exceeds tolerance. `EXECUTE_REJECTED_IK_FOR_DIAGNOSTIC=True` is a conspicuous teaching override: it may execute only a finite, in-limit, finger-preserving candidate, while `ik_valid` remains false.

The diagnostic records actual hand motion and remaining error to the requested target. A final fixed-camera frame, when enabled, still shows only scene state; the red marker remains the reachable baseline marker and may not represent the out-of-view requested target.


In [ ]:
franka.set_dofs_position(q_start, dofs_idx_local=all_dofs, zero_velocity=True)
if render_enabled:
    wrist_camera.move_to_attach()
unreachable_start_position, unreachable_start_quaternion = (
    value.reshape(-1) for value in read_hand_pose(hand)
)

q_unreachable_raw, unreachable_error_raw = franka.inverse_kinematics(
    link=hand,
    pos=UNREACHABLE_POSITION,
    quat=target_quaternion,
    init_qpos=q_start,
    respect_joint_limit=True,
    max_samples=50,
    max_solver_iters=20,
    damping=0.01,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    pos_mask=POSITION_MASK.tolist(),
    rot_mask=ROTATION_MASK.tolist(),
    dofs_idx_local=arm_dofs,
    return_error=True,
)
unreachable_ik = assess_single_ik(
    q_unreachable_raw,
    unreachable_error_raw,
    q_start[finger_dofs],
)
unreachable_q = unreachable_ik["candidate"]
unreachable_rejection_reasons = [
    name for name, passed in unreachable_ik["checks"].items() if not passed
]
unreachable_structurally_safe = all(
    unreachable_ik["checks"].get(name, False)
    for name in (
        "q_shape",
        "residual_shape",
        "q_finite",
        "residual_finite",
        "joint_limits",
        "fingers_preserved",
    )
)
diagnostic_override = bool(
    EXECUTE_REJECTED_IK_FOR_DIAGNOSTIC
    and unreachable_structurally_safe
    and not unreachable_ik["valid"]
)
unreachable_command_executed = False
unreachable_position_history = [unreachable_start_position]
unreachable_final_position = np.full(3, np.nan)
unreachable_hand_motion = np.nan
unreachable_remaining_error = np.nan
unreachable_fixed_rgb = None

print("unreachable candidate shape:", unreachable_q.shape)
print("unreachable residual:", unreachable_ik["residual"])
print("unreachable position residual [m]:", unreachable_ik["position_norm"])
print("unreachable rotation residual [rad]:", unreachable_ik["rotation_norm"])
print("IK valid:", unreachable_ik["valid"])
print("normal control decision: REJECT")
print("rejection reasons:", unreachable_rejection_reasons)

if diagnostic_override:
    print("WARNING — executing a rejected finite candidate for diagnosis only")
    unreachable_command_executed = True
    for _ in range(REACH_STEPS):
        franka.control_dofs_position(unreachable_q, dofs_idx_local=all_dofs)
        scene.step()
        if render_enabled:
            wrist_camera.move_to_attach()
        position, _ = read_hand_pose(hand)
        unreachable_position_history.append(position.reshape(-1))
    unreachable_final_position = unreachable_position_history[-1]
    unreachable_hand_motion = float(
        np.linalg.norm(unreachable_final_position - unreachable_start_position)
    )
    unreachable_remaining_error = float(
        np.linalg.norm(unreachable_final_position - UNREACHABLE_POSITION)
    )
    if render_enabled:
        unreachable_fixed_rgb, _ = render_observation(
            fixed_camera,
            "fixed camera rejected diagnostic",
            include_depth=False,
        )
elif unreachable_ik["valid"]:
    raise AssertionError("the deliberately unreachable target unexpectedly passed acceptance")
else:
    print("diagnostic command skipped because the candidate was not structurally safe")

unreachable_position_history = np.asarray(unreachable_position_history, dtype=float)
print("diagnostic command executed:", unreachable_command_executed)
print("actual hand motion [m]:", unreachable_hand_motion)
print("remaining requested position error [m]:", unreachable_remaining_error)


In [ ]:
reachable_status = "accepted and dynamically executed"
unreachable_status = (
    "rejected; executed only under diagnostic override"
    if unreachable_command_executed
    else "rejected; not executed"
)
comparison_lines = [
    "| Metric | Reachable target | Unreachable target |",
    "|---|---:|---:|",
    f"| IK position residual [m] | {reachable_ik['position_norm']:.6f} | {unreachable_ik['position_norm']:.6f} |",
    f"| IK rotation residual [rad] | {reachable_ik['rotation_norm']:.6f} | {unreachable_ik['rotation_norm']:.6f} |",
    f"| Candidate finite | {reachable_ik['checks']['q_finite']} | {unreachable_ik['checks']['q_finite']} |",
    f"| IK accepted | {reachable_ik['valid']} | {unreachable_ik['valid']} |",
    f"| Execution status | {reachable_status} | {unreachable_status} |",
    f"| Final requested position error [m] | {reachable_final_position_error:.6f} | {unreachable_remaining_error:.6f} |",
]
display(Markdown("\n".join(comparison_lines)))

diagnostic_figure = plt.figure(figsize=(9, 6))
diagnostic_axis = diagnostic_figure.add_subplot(111, projection="3d")
diagnostic_axis.plot(
    unreachable_position_history[:, 0],
    unreachable_position_history[:, 1],
    unreachable_position_history[:, 2],
    color="#7A5195",
    label="measured diagnostic trajectory",
)
diagnostic_axis.scatter(
    *unreachable_start_position,
    color="#E9C46A",
    s=60,
    label="hand before override",
)
if unreachable_command_executed:
    diagnostic_axis.scatter(
        *unreachable_final_position,
        color="#2A9D5B",
        s=60,
        label="hand after override",
    )
diagnostic_axis.scatter(
    *UNREACHABLE_POSITION,
    color="#D95F43",
    s=100,
    marker="*",
    label="requested unreachable target",
)
diagnostic_axis.set(
    xlabel="world x [m]",
    ylabel="world y [m]",
    zlabel="world z [m]",
    title="Measured-state unreachable diagnostic — not a camera frame",
)
diagnostic_axis.legend()
diagnostic_figure.tight_layout()
diagnostic_path = output_dir / "l05_unreachable_diagnostic.png"
diagnostic_figure.savefig(diagnostic_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(diagnostic_figure)

if render_enabled:
    rejected_figure, rejected_axis = plt.subplots(figsize=(7, 4.5))
    rejected_axis.imshow(unreachable_fixed_rgb)
    rejected_axis.set_title(
        "Genesis fixed camera — rejected best-effort q\n"
        "red marker remains the reachable baseline marker"
    )
    rejected_axis.axis("off")
    rejected_figure.tight_layout()
    rejected_path = output_dir / "l05_rejected_camera_frame.png"
    rejected_figure.savefig(rejected_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(rejected_figure)
    print("saved:", rejected_path.resolve())

unreachable_interpretation = f"""
### Guided unreachable interpretation

**1. Acceptance result.** The finite candidate check is
{unreachable_ik['checks']['q_finite']}, but `ik_valid` is
{unreachable_ik['valid']}. The failed conditions are
{', '.join(unreachable_rejection_reasons)}. Passing structural checks does not
cancel a residual rejection.

**2. Reachable versus unreachable residuals.** Position residual changes from
{reachable_ik['position_norm']:.6f} to
{unreachable_ik['position_norm']:.6f} m; rotation residual changes from
{reachable_ik['rotation_norm']:.6f} to
{unreachable_ik['rotation_norm']:.6f} rad. Each component is judged against its
own enabled tolerance.

**3. Diagnostic execution.** The rejected candidate was
{'executed under the explicit override' if unreachable_command_executed else 'not executed'}.
Measured hand motion is {unreachable_hand_motion:.6f} m and remaining error to
the requested target is {unreachable_remaining_error:.6f} m. Those values
describe best effort, not successful reachability.

**4. Evidence boundary.** A finite q, visible motion, and no exception do not
prove IK convergence, collision-free motion, grasp success, or task success.
The camera frame, when enabled, shows scene state but does not replace residual
or measured-error checks.
"""
display(Markdown(unreachable_interpretation))
print("saved measured-state diagnostic:", diagnostic_path.resolve())


## Part C · B=4 reaching and selective updates

A second, camera-free Scene builds four environments from one Plane-and-Franka topology. The baseline uses target positions `(4, 3)`, quaternions `(4, 4)`, IK q `(4, 9)`, residuals `(4, 6)`, and measured hand positions `(4, 3)`. Every row must pass acceptance and dynamic error checks; a mean cannot hide failure.

The selective experiment sends exactly two target/q rows to `envs_idx=[1, 3]`. Environments 0 and 2 receive no new target, but they are not frozen: their previous PD targets remain active. The checks therefore combine selected-target error, untouched motion, and retained-target error change.

This extension teaches batch shape and row-wise evidence. Throughput and data recording remain in L08, and a single camera frame is not used as evidence for all four environments.


In [ ]:
BATCH_SIZE = 4
BATCH_STEPS = 220
BATCH_POSITION_THRESHOLD = 0.08

batch_scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=DT, substeps=SUBSTEPS),
    rigid_options=gs.options.RigidOptions(enable_collision=True),
    show_viewer=False,
)
batch_scene.add_entity(gs.morphs.Plane())
batch_franka = batch_scene.add_entity(gs.morphs.MJCF(file=FRANKA_MJCF))
batch_scene.build(n_envs=BATCH_SIZE, env_spacing=(1.1, 1.1))
batch_hand = batch_franka.get_link("hand")

batch_dof_indices = []
for name in JOINT_NAMES:
    joint = batch_franka.get_joint(name)
    if joint.n_dofs != 1:
        raise AssertionError(f"{name}: expected one DOF")
    batch_dof_indices.extend(joint.dofs_idx_local)
batch_all_dofs = np.asarray(batch_dof_indices, dtype=int)
batch_arm_dofs = batch_all_dofs[:7]
batch_finger_dofs = batch_all_dofs[7:]
batch_q_start = np.tile(q_start, (BATCH_SIZE, 1))

batch_franka.set_dofs_kp(kp, dofs_idx_local=batch_all_dofs)
batch_franka.set_dofs_kv(kv, dofs_idx_local=batch_all_dofs)
batch_franka.set_dofs_force_range(
    force_lower,
    force_upper,
    dofs_idx_local=batch_all_dofs,
)
batch_franka.set_dofs_position(
    batch_q_start,
    dofs_idx_local=batch_all_dofs,
    zero_velocity=True,
)

batch_qpos_initial = read_dofs(batch_franka, batch_all_dofs)
batch_initial_positions, _ = read_hand_pose(batch_hand)
batch_target_positions = np.array(
    [
        [0.42, -0.12, 0.35],
        [0.48, -0.04, 0.40],
        [0.48, 0.06, 0.32],
        [0.40, 0.14, 0.38],
    ],
    dtype=float,
)
batch_target_quaternions = np.tile(target_quaternion, (BATCH_SIZE, 1))
batch_initial_errors = np.linalg.norm(
    batch_initial_positions - batch_target_positions,
    axis=1,
)

batch_q_goal_raw, batch_ik_error_raw = batch_franka.inverse_kinematics(
    link=batch_hand,
    pos=batch_target_positions,
    quat=batch_target_quaternions,
    init_qpos=batch_q_start,
    respect_joint_limit=True,
    max_samples=50,
    max_solver_iters=20,
    damping=0.01,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    pos_mask=POSITION_MASK.tolist(),
    rot_mask=ROTATION_MASK.tolist(),
    dofs_idx_local=batch_arm_dofs,
    return_error=True,
)
batch_q_goal = to_numpy(batch_q_goal_raw).astype(float)
batch_ik_error = to_numpy(batch_ik_error_raw).astype(float)

batch_limit_lower_raw, batch_limit_upper_raw = (
    to_numpy(value).astype(float)
    for value in batch_franka.get_dofs_limit(dofs_idx_local=batch_all_dofs)
)
batch_limit_lower = np.broadcast_to(batch_limit_lower_raw, (BATCH_SIZE, 9))
batch_limit_upper = np.broadcast_to(batch_limit_upper_raw, (BATCH_SIZE, 9))
batch_position_residuals = np.linalg.norm(
    batch_ik_error[:, :3][:, POSITION_MASK],
    axis=1,
)
batch_rotation_residuals = np.linalg.norm(
    batch_ik_error[:, 3:][:, ROTATION_MASK],
    axis=1,
)
batch_candidate_checks = {
    "initial_qpos_shape": batch_qpos_initial.shape == (BATCH_SIZE, 9),
    "target_position_shape": batch_target_positions.shape == (BATCH_SIZE, 3),
    "target_quaternion_shape": batch_target_quaternions.shape == (BATCH_SIZE, 4),
    "candidate_shape": batch_q_goal.shape == (BATCH_SIZE, 9),
    "residual_shape": batch_ik_error.shape == (BATCH_SIZE, 6),
    "candidate_and_residual_finite": (
        np.isfinite(batch_q_goal).all() and np.isfinite(batch_ik_error).all()
    ),
    "joint_limits": (
        np.all(batch_q_goal >= batch_limit_lower - LIMIT_TOLERANCE)
        and np.all(batch_q_goal <= batch_limit_upper + LIMIT_TOLERANCE)
    ),
    "fingers_preserved": np.allclose(
        batch_q_goal[:, batch_finger_dofs],
        batch_q_start[:, batch_finger_dofs],
        rtol=0.0,
        atol=1e-7,
    ),
    "position_residuals": np.all(
        batch_position_residuals <= IK_POSITION_TOLERANCE
    ),
    "rotation_residuals": np.all(
        batch_rotation_residuals <= IK_ROTATION_TOLERANCE
    ),
}
if not all(batch_candidate_checks.values()):
    raise AssertionError(batch_candidate_checks)

for _ in range(BATCH_STEPS):
    batch_franka.control_dofs_position(
        batch_q_goal,
        dofs_idx_local=batch_all_dofs,
    )
    batch_scene.step()

batch_qpos_final = read_dofs(batch_franka, batch_all_dofs)
batch_measured_positions, _ = read_hand_pose(batch_hand)
batch_final_errors = np.linalg.norm(
    batch_measured_positions - batch_target_positions,
    axis=1,
)
batch_execution_checks = {
    "final_qpos_shape": batch_qpos_final.shape == (BATCH_SIZE, 9),
    "measured_position_shape": batch_measured_positions.shape == (BATCH_SIZE, 3),
    "state_finite": (
        np.isfinite(batch_qpos_final).all()
        and np.isfinite(batch_measured_positions).all()
    ),
    "every_error_below_threshold": np.all(
        batch_final_errors < BATCH_POSITION_THRESHOLD
    ),
    "every_error_decreased": np.all(batch_final_errors < batch_initial_errors),
}
if not all(batch_execution_checks.values()):
    raise AssertionError(batch_execution_checks)

batch_lines = [
    "| Env | IK pos residual [m] | IK rot residual [rad] | Initial error [m] | Final error [m] | Accepted |",
    "|---:|---:|---:|---:|---:|---|",
]
for env_index in range(BATCH_SIZE):
    batch_lines.append(
        f"| {env_index} | {batch_position_residuals[env_index]:.6f} | "
        f"{batch_rotation_residuals[env_index]:.6f} | "
        f"{batch_initial_errors[env_index]:.6f} | "
        f"{batch_final_errors[env_index]:.6f} | yes |"
    )
display(Markdown("\n".join(batch_lines)))
print("batch candidate checks:", batch_candidate_checks)
print("batch execution checks:", batch_execution_checks)


In [ ]:
SELECTED_ENVS = np.array([1, 3], dtype=int)
UNTOUCHED_ENVS = np.array([0, 2], dtype=int)
SELECTIVE_TARGET_POSITIONS = np.array(
    [
        [0.44, -0.16, 0.32],
        [0.50, 0.12, 0.40],
    ],
    dtype=float,
)
SELECTIVE_TARGET_QUATERNIONS = np.tile(
    target_quaternion,
    (len(SELECTED_ENVS), 1),
)
SELECTED_POSITION_THRESHOLD = 0.08
UNTOUCHED_MOTION_THRESHOLD = 0.005
UNTOUCHED_ERROR_WORSENING_TOLERANCE = 0.002

positions_before_selective, _ = read_hand_pose(batch_hand)
q_before_selective = read_dofs(batch_franka, batch_all_dofs)
selected_initial_errors = np.linalg.norm(
    positions_before_selective[SELECTED_ENVS] - SELECTIVE_TARGET_POSITIONS,
    axis=1,
)

q_selective_raw, selective_ik_error_raw = batch_franka.inverse_kinematics(
    link=batch_hand,
    pos=SELECTIVE_TARGET_POSITIONS,
    quat=SELECTIVE_TARGET_QUATERNIONS,
    respect_joint_limit=True,
    max_samples=50,
    max_solver_iters=20,
    damping=0.01,
    pos_tol=IK_POSITION_TOLERANCE,
    rot_tol=IK_ROTATION_TOLERANCE,
    pos_mask=POSITION_MASK.tolist(),
    rot_mask=ROTATION_MASK.tolist(),
    dofs_idx_local=batch_arm_dofs,
    return_error=True,
    envs_idx=SELECTED_ENVS,
)
q_selective = to_numpy(q_selective_raw).astype(float)
selective_ik_error = to_numpy(selective_ik_error_raw).astype(float)
selective_position_residuals = np.linalg.norm(
    selective_ik_error[:, :3][:, POSITION_MASK],
    axis=1,
)
selective_rotation_residuals = np.linalg.norm(
    selective_ik_error[:, 3:][:, ROTATION_MASK],
    axis=1,
)
selected_lower = batch_limit_lower[SELECTED_ENVS]
selected_upper = batch_limit_upper[SELECTED_ENVS]
selective_candidate_checks = {
    "selected_target_shape": SELECTIVE_TARGET_POSITIONS.shape == (2, 3),
    "selected_quaternion_shape": SELECTIVE_TARGET_QUATERNIONS.shape == (2, 4),
    "selected_q_shape": q_selective.shape == (2, 9),
    "selected_residual_shape": selective_ik_error.shape == (2, 6),
    "selected_values_finite": (
        np.isfinite(q_selective).all() and np.isfinite(selective_ik_error).all()
    ),
    "selected_joint_limits": (
        np.all(q_selective >= selected_lower - LIMIT_TOLERANCE)
        and np.all(q_selective <= selected_upper + LIMIT_TOLERANCE)
    ),
    "selected_fingers_preserved": np.allclose(
        q_selective[:, batch_finger_dofs],
        q_before_selective[SELECTED_ENVS][:, batch_finger_dofs],
        rtol=0.0,
        atol=1e-7,
    ),
    "selected_position_residuals": np.all(
        selective_position_residuals <= IK_POSITION_TOLERANCE
    ),
    "selected_rotation_residuals": np.all(
        selective_rotation_residuals <= IK_ROTATION_TOLERANCE
    ),
}
if not all(selective_candidate_checks.values()):
    raise AssertionError(selective_candidate_checks)

for _ in range(BATCH_STEPS):
    batch_franka.control_dofs_position(
        q_selective,
        dofs_idx_local=batch_all_dofs,
        envs_idx=SELECTED_ENVS,
    )
    batch_scene.step()

positions_after_selective, _ = read_hand_pose(batch_hand)
selected_final_errors = np.linalg.norm(
    positions_after_selective[SELECTED_ENVS] - SELECTIVE_TARGET_POSITIONS,
    axis=1,
)
retained_target_errors = np.linalg.norm(
    positions_after_selective[UNTOUCHED_ENVS]
    - batch_target_positions[UNTOUCHED_ENVS],
    axis=1,
)
retained_error_change = retained_target_errors - batch_final_errors[UNTOUCHED_ENVS]
untouched_motion = np.linalg.norm(
    positions_after_selective[UNTOUCHED_ENVS]
    - positions_before_selective[UNTOUCHED_ENVS],
    axis=1,
)
selective_execution_checks = {
    "all_positions_shape": positions_after_selective.shape == (BATCH_SIZE, 3),
    "all_positions_finite": np.isfinite(positions_after_selective).all(),
    "selected_error_threshold": np.all(
        selected_final_errors < SELECTED_POSITION_THRESHOLD
    ),
    "selected_errors_decreased": np.all(
        selected_final_errors < selected_initial_errors
    ),
    "untouched_retained_error_threshold": np.all(
        retained_target_errors < BATCH_POSITION_THRESHOLD
    ),
    "untouched_motion_threshold": np.all(
        untouched_motion < UNTOUCHED_MOTION_THRESHOLD
    ),
    "untouched_error_not_materially_worse": np.all(
        retained_error_change <= UNTOUCHED_ERROR_WORSENING_TOLERANCE
    ),
}
if not all(selective_execution_checks.values()):
    raise AssertionError(selective_execution_checks)

selective_lines = [
    "| Env | Role | Error before update [m] | Error after update [m] | Motion during update [m] |",
    "|---:|---|---:|---:|---:|",
]
for row, env_index in enumerate(SELECTED_ENVS):
    selective_lines.append(
        f"| {env_index} | selected: new target | "
        f"{selected_initial_errors[row]:.6f} | {selected_final_errors[row]:.6f} | "
        f"{np.linalg.norm(positions_after_selective[env_index] - positions_before_selective[env_index]):.6f} |"
    )
for row, env_index in enumerate(UNTOUCHED_ENVS):
    selective_lines.append(
        f"| {env_index} | untouched: retained target | "
        f"{batch_final_errors[env_index]:.6f} | {retained_target_errors[row]:.6f} | "
        f"{untouched_motion[row]:.6f} |"
    )
display(Markdown("\n".join(selective_lines)))
print("selective candidate checks:", selective_candidate_checks)
print("selective execution checks:", selective_execution_checks)


In [ ]:
batch_figure = plt.figure(figsize=(12, 5))
baseline_axis = batch_figure.add_subplot(121, projection="3d")
selective_axis = batch_figure.add_subplot(122, projection="3d")

for env_index in range(BATCH_SIZE):
    baseline_axis.plot(
        [batch_target_positions[env_index, 0], batch_measured_positions[env_index, 0]],
        [batch_target_positions[env_index, 1], batch_measured_positions[env_index, 1]],
        [batch_target_positions[env_index, 2], batch_measured_positions[env_index, 2]],
        "-o",
        label=f"env {env_index}",
    )
baseline_axis.set(
    xlabel="world x [m]",
    ylabel="world y [m]",
    zlabel="world z [m]",
    title="B=4 baseline: target → measured",
)
baseline_axis.legend(fontsize=8)

for row, env_index in enumerate(SELECTED_ENVS):
    selective_axis.plot(
        [SELECTIVE_TARGET_POSITIONS[row, 0], positions_after_selective[env_index, 0]],
        [SELECTIVE_TARGET_POSITIONS[row, 1], positions_after_selective[env_index, 1]],
        [SELECTIVE_TARGET_POSITIONS[row, 2], positions_after_selective[env_index, 2]],
        "-o",
        label=f"selected env {env_index}",
    )
for row, env_index in enumerate(UNTOUCHED_ENVS):
    selective_axis.plot(
        [positions_before_selective[env_index, 0], positions_after_selective[env_index, 0]],
        [positions_before_selective[env_index, 1], positions_after_selective[env_index, 1]],
        [positions_before_selective[env_index, 2], positions_after_selective[env_index, 2]],
        "--o",
        label=f"untouched env {env_index}",
    )
selective_axis.set(
    xlabel="world x [m]",
    ylabel="world y [m]",
    zlabel="world z [m]",
    title="Selective update: selected targets and untouched motion",
)
selective_axis.legend(fontsize=8)
batch_figure.suptitle("Measured-state batch evidence — not camera frames")
batch_figure.tight_layout()
batch_path = output_dir / "l05_batch_measured_state.png"
batch_figure.savefig(batch_path, dpi=140, bbox_inches="tight")
plt.show()
plt.close(batch_figure)

baseline_error_text = ", ".join(
    f"env {index}: {batch_initial_errors[index]:.4f}→{batch_final_errors[index]:.4f} m"
    for index in range(BATCH_SIZE)
)
selected_error_text = ", ".join(
    f"env {env_index}: {selected_initial_errors[row]:.4f}→{selected_final_errors[row]:.4f} m"
    for row, env_index in enumerate(SELECTED_ENVS)
)
untouched_text = ", ".join(
    f"env {env_index}: motion {untouched_motion[row]:.4f} m, "
    f"retained-error change {retained_error_change[row]:+.4f} m"
    for row, env_index in enumerate(UNTOUCHED_ENVS)
)
batch_interpretation = f"""
### Guided batch interpretation

**1. Shapes and IK acceptance.** B={BATCH_SIZE} produced qpos
{batch_qpos_initial.shape}, targets {batch_target_positions.shape}, candidates
{batch_q_goal.shape}, and residuals {batch_ik_error.shape}. Every row passed
finiteness, limits, finger preservation, and both residual tolerances.

**2. Baseline execution.** {baseline_error_text}. Every conclusion is
per-environment; the mean is not used to hide a failed row.

**3. Selected environments.** `envs_idx={SELECTED_ENVS.tolist()}` receives
exactly {len(SELECTED_ENVS)} target and command rows. {selected_error_text}.
Each selected row is checked against its own new target.

**4. Untouched environments.** {untouched_text}. Environments
{UNTOUCHED_ENVS.tolist()} received no new target, but they continued evolving
under their retained PD targets. Small motion is therefore checked together
with retained-target error rather than called cross-environment interference.

This plot is measured-state evidence, not a camera frame. Throughput and data
recording remain topics for L08.
"""
display(Markdown(batch_interpretation))
print("saved:", batch_path.resolve())


## Checkpoint before the final checks

Use the generated tables, trajectories, camera/fallback figures, and Guided interpretations:

1. Why do target and measured hand pose need a common frame?
2. Why are w-x-y-z order, normalization, and `q`/`-q` three separate checks?
3. How do solver residual, FK prediction, and dynamic tracking error differ?
4. Why is a `(6, 7)` Jacobian statement local to one configuration?
5. Why does damped IK neither guarantee reachability nor plan a collision-free path?
6. What proves the wrist camera followed the hand?
7. Why does finite far-plane depth not automatically represent a surface?
8. Why may untouched batch environments still move?

Explain each answer in your own words before running the final cell.


In [ ]:
runtime_checks = {
    "genesis_1_3_3": environment["genesis_world"] == "1.3.3",
    "actual_backend_supported": actual_backend in {"cpu", "amdgpu"},
    "forced_cpu_honored": backend_mode != "cpu" or actual_backend == "cpu",
    "timing_positive": DT > 0 and SUBSTEPS > 0 and REACH_STEPS == 180,
}
unreachable_checks = {
    "unreachable_candidate_rejected": not unreachable_ik["valid"],
    "unreachable_has_rejection_reason": bool(unreachable_rejection_reasons),
    "diagnostic_override_explicit": (
        not unreachable_command_executed or diagnostic_override
    ),
    "diagnostic_motion_finite": (
        not unreachable_command_executed
        or (
            np.isfinite(unreachable_hand_motion)
            and np.isfinite(unreachable_remaining_error)
        )
    ),
    "requested_target_not_reached": (
        not unreachable_command_executed
        or unreachable_remaining_error > IK_POSITION_TOLERANCE
    ),
}
all_checks = {
    **runtime_checks,
    **structure_checks,
    **kinematic_checks,
    **reachable_ik["checks"],
    **solver_state_checks,
    **execution_checks,
    **camera_checks,
    **unreachable_checks,
    **batch_candidate_checks,
    **batch_execution_checks,
    **selective_candidate_checks,
    **selective_execution_checks,
}
for name, passed in all_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

failed_checks = [name for name, passed in all_checks.items() if not passed]
print("requested backend:", backend_mode)
print("actual backend:", actual_backend)
print(
    "rendering:",
    (
        "PASSED — Genesis fixed RGB/depth and wrist RGB captured"
        if render_enabled
        else "SKIP — disabled before build; measured-state schematics used"
    ),
)
print("evidence directory:", output_dir.resolve())
if failed_checks:
    raise AssertionError("L05 checks failed: " + ", ".join(failed_checks))

evidence_summary = f"""
### Evidence summary generated by this run

- Reachable IK: candidate accepted={reachable_ik['valid']}; solver position /
  rotation residuals are {reachable_ik['position_norm']:.6f} m /
  {reachable_ik['rotation_norm']:.6f} rad.
- Dynamic reach: measured position error is
  {reachable_initial_position_error:.6f}→{reachable_final_position_error:.6f} m;
  orientation error is
  {reachable_initial_orientation_error:.6f}→{reachable_final_orientation_error:.6f} rad.
- Unreachable case: accepted={unreachable_ik['valid']}; diagnostic
  executed={unreachable_command_executed}; remaining position error is
  {unreachable_remaining_error:.6f} m.
- Camera path:
  {'validated fixed RGB/depth, wrist RGB, and fresh extrinsics' if render_enabled else 'SKIP — no camera was created or claimed'}.
- B=4 baseline and selective updates passed every per-environment threshold;
  no batch mean was used as a substitute for row-wise acceptance.

L06 can reuse the frame and camera contracts in a tabletop scene. L07 can
sequence pose targets but must add motion and grasp logic. L08 can reuse the
batch shapes for synchronized recording and throughput analysis.
"""
display(Markdown(evidence_summary))
print("L05 CHECK: PASSED")
